# This lesson demonstrates how to implement neural networks for both regression and classification tasks using two popular deep learning frameworks: PyTorch and TensorFlow. 

In [3]:
!pip install pyspark torch torchvision

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from keras import layers, Input
from keras.utils import to_categorical

# Part 1: MLP for Regression – Predicting California Housing Prices

The lesson begins with loading the California Housing dataset, which contains information about housing prices along with various features like location, number of rooms, etc.

The data preparation follows standard machine learning practices:

Loading the dataset into a Pandas DataFrame
Extracting feature matrices (X) and target values (y)
Normalizing features using StandardScaler to help the neural network train more efficiently
Splitting data into training (80%) and testing (20%) sets

# Step 1: Load and prepare the Data

#### Load the California housing dataset into a Pandas DataFrame.
#### Convert it into a Spark DataFrame for distributed processing.
#### Assemble feature vectors for PySpark.

In [10]:
# Load and Normalize California Housing Data
housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
df_housing['target'] = housing.target  # Apply log transformation

X = df_housing[housing.feature_names].values
y = df_housing['target'].values
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Step 2: PyTorch convert Spark Data to Pytorch Tensors

In [13]:
# Convert NumPy arrays to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Step 3: PyTorch Build the MLP Model for Regression

In [18]:
# Define MLP architecture to match TensorFlow
class MLPRegressor(nn.Module):
    def __init__(self, input_size):
        super(MLPRegressor, self).__init__()
        # First layer with BatchNorm
        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )
        # Second layer with ReLU and then Dropout (matching TensorFlow placement)
        self.layer2 = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3)  # Moved dropout after ReLU to match TensorFlow
        )
        # Output layer
        self.layer3 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

# Create model
pt_model_regression = MLPRegressor(X_train.shape[1])

# Use Adam optimizer with default settings to match TensorFlow
# TensorFlow default Adam: lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7
optimizer = optim.Adam(
    pt_model_regression.parameters(),
    lr=0.001,  # Default TensorFlow learning rate
    betas=(0.9, 0.999),  # Default betas
    eps=1e-8  # PyTorch default epsilon (slightly different from 
)

# Step 4: PyTorch- Train the Regression Model

In [21]:
# Loss function
criterion = nn.MSELoss()

# Create DataLoader for mini-batch training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Match TF batch size=32

# Training loop with mini-batches (50 epochs to match TensorFlow)
num_epochs = 50
for epoch in range(num_epochs):
    running_loss = 0.0

    # Mini-batch training
    for batch_x, batch_y in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = pt_model_regression(batch_x)
        loss = criterion(outputs, batch_y)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Print statistics (similar to TensorFlow verbose=1)
    if epoch % 10 == 0 or epoch == num_epochs-1:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

Epoch [1/50], Loss: 1.0466
Epoch [11/50], Loss: 0.4292
Epoch [21/50], Loss: 0.3920
Epoch [31/50], Loss: 0.3867
Epoch [41/50], Loss: 0.3712
Epoch [50/50], Loss: 0.3709


# Step 5: PyTorch- Evaluate the Regresssion Model

In [24]:
# Evaluate model
pt_model_regression.eval()
with torch.no_grad():
    predictions = pt_model_regression(X_test_t)
    mse_loss = criterion(predictions, y_test_t)
    mae_loss = nn.L1Loss()(predictions, y_test_t)  # Calculate MAE to match TensorFlow metrics

print(f'Test Mean Squared Error: {mse_loss.item():.4f}')
print(f'Test Mean Absolute Error: {mae_loss.item():.4f}')

Test Mean Squared Error: 0.3179
Test Mean Absolute Error: 0.3779


# Step 6: PyTorch: Make Predictions with the Regression Model

In [27]:
# Make predictions with scaled input (matching TensorFlow's approach)
sample_input_np = np.array([[8.0, 41.0, 6.0, 1.0, 950.0, 4.0, 37.0, -122.0]])
sample_input_scaled = scaler.transform(sample_input_np)  # Apply scaling
sample_input_t = torch.tensor(sample_input_scaled, dtype=torch.float32)

pt_model_regression.eval()
with torch.no_grad():
    pt_prediction = pt_model_regression(sample_input_t).item()

print(f'Predicted House Price: {pt_prediction:.2f}')

Predicted House Price: 3.42


# Step 7: TensorFlow Build the MLP Model for Regression

In [30]:
# Define MLP architecture
tf_model_regression = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1)
])

# Compile the model
tf_model_regression.compile(optimizer="adam", loss="mse", metrics=["mae"])

# Display model summary
tf_model_regression.summary()

C:\Users\12146\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 128 (512.00 B)

# Step 8: TensorFlow- Train the Regression Model

In [33]:
# Train the model
tf_history_regression = tf_model_regression.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

Epoch 1/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.0966 - mae: 0.7729 - val_loss: 0.5911 - val_mae: 0.5289
Epoch 2/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6548 - mae: 0.5902 - val_loss: 0.4763 - val_mae: 0.4803
Epoch 3/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5633 - mae: 0.5444 - val_loss: 0.4818 - val_mae: 0.4853
Epoch 4/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5089 - mae: 0.5152 - val_loss: 0.5561 - val_mae: 0.5199
Epoch 5/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4835 - mae: 0.4961 - val_loss: 0.3971 - val_mae: 0.4254
Epoch 6/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4524 - mae: 0.4763 - val_loss: 0.4811 - val_mae: 0.4862
Epoch 7/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4395 - mae: 0.4696 - val_loss: 0.4492 - val_mae: 0.4601
Epoch 8/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4166 - mae: 0.4535 - val_loss: 0.3707 - val_mae: 0.4067
Epoch 9/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - lo

# Step 9: TensorFlow - Evaluate the Regression Model

In [36]:
# Evaluate the model on test data
loss, mae = tf_model_regression.evaluate(X_test, y_test)
print(f"Test Mean Absolute Error: {mae:.4f}")

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3672 - mae: 0.4333
Test Mean Absolute Error: 0.4333


# Part 1: Implementation for Regression - Pytorch and TensorFlow

#### Part 1: Implementations for MLP for Regression with PyTorch– Predicting California Housing Prices

##### PyTorch requires data to be in tensor format, so the NumPy arrays are converted to PyTorch tensors with appropriate data types (float32 for features and targets).

#### Step 3- Building the Model
#### The PyTorch model is defined as a custom class MLPRegressor that inherits from nn.Module. The architecture consists of:

#### First layer: Linear(input_size, 64) → BatchNorm → ReLU
#### Second layer: Linear(64, 32) → ReLU → Dropout(0.3)
#### Output layer: Linear(32, 1)
#### This architecture is specifically designed to match the TensorFlow implementation for fair comparison.

#### Step 4- Training the Model
#### Training in PyTorch requires manually implementing the training loop:

#### Define loss function (MSE) and optimizer (Adam)
#### Create a DataLoader for batching data
#### For each epoch:
#### Loop through mini-batches
#### Zero out gradients
#### Forward pass to get predictions
#### Calculate loss
#### Backward pass to compute gradients
#### Update weights
#### The training output shows the loss decreasing from 1.1118 to 0.3757 over 50 epochs, indicating the model is learning.

#### Evaluating the Model
#### The model achieves a test MSE of 0.3917 and MAE of 0.4252, which are respectable metrics for this dataset.

#### Step 6 Making predictions:
#### A single sample house is used to demonstrate prediction, resulting in a predicted price of 3.55 (likely in $100,000s).

#### Part 1: Implementations for MLP for Regression with TensorFlow Implementation- Predicting California Housing Prices

### Step 7: Building the Model
#### TensorFlow's model is created using the Sequential API, which provides a more concise syntax:

In [60]:
tf_model_regression = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1)
])

C:\Users\12146\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#### The architecture matches the PyTorch model for comparison purposes.

#### Step 8 : Training the Model


In [68]:
tf_model_regression .compile(optimizer='adam', loss='mse')

#### Unlike PyTorch, TensorFlow provides a high-level fit() method that handles the training loop internally:

In [70]:
tf_history_regression = tf_model_regression.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

Epoch 1/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1.1605 - val_loss: 0.5942
Epoch 2/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6381 - val_loss: 0.5086
Epoch 3/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5598 - val_loss: 0.4360
Epoch 4/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5161 - val_loss: 0.3988
Epoch 5/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4807 - val_loss: 0.3847
Epoch 6/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4606 - val_loss: 0.3794
Epoch 7/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4460 - val_loss: 0.3686
Epoch 8/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4276 - val_loss: 0.3675
Epoch 9/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4270 - val_loss: 0.3668
Epoch 10/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4151 - val_loss: 0.4037
Epoch 11/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4085 - val_loss: 0.3377
Epoch 12/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

##### This single call handles all the operations that were manually implemented in PyTorch.

# Part 2 MLP for lassification - Predicting Iris Flowe Species

#### Step 11 Load and Prepare the Data
#### Load the Iris dataset into a Pandas DataFrame.
#### Convert it into a Spark DataFrame.
#### Encode categorical labels and assemble features.

In [78]:
# Load dataset
iris = load_iris()
df_iris = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_iris['label'] = iris.target

# Convert to NumPy arrays
X = df_iris[iris.feature_names].values
y = df_iris['label'].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Step 12- Pytorch Convert Spark Data to PyTorch Tensors

In [81]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)  # Long tensor for classification
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

#### Step 13- PyTorch Build the MLP Model for Classification

In [89]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

# Initialize model
input_size = X_train_t.shape[1]
hidden_size = 16
output_size = len(iris.target_names)

pt_model_classification = MLPClassifier(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model_classification.parameters(), lr=0.01)

#### Step 14 PyTorch - Train the Classification Model

In [92]:
num_epochs = 100
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = pt_model_classification(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [0/100], Loss: 1.1124
Epoch [10/100], Loss: 0.9202
Epoch [20/100], Loss: 0.7999
Epoch [30/100], Loss: 0.7443
Epoch [40/100], Loss: 0.7101
Epoch [50/100], Loss: 0.6765
Epoch [60/100], Loss: 0.6430
Epoch [70/100], Loss: 0.6180
Epoch [80/100], Loss: 0.6033
Epoch [90/100], Loss: 0.5947


#### Step 15: PyTorch Evaluate the Classification Model

In [97]:
with torch.no_grad():
    predictions = pt_model_classification(X_test_t)
    predicted_labels = torch.argmax(predictions, axis=1)
    accuracy = (predicted_labels == y_test_t).sum().item() / y_test_t.size(0)

print(f'Classification Accuracy: {accuracy:.4f}')

Classification Accuracy: 1.0000


#### Step 16: PyTorch - Make Predictions with the Classification Model

In [100]:
sample_input = torch.tensor([[6.7, 3.1, 4.9, 1.5]])  # Example from dataset
with torch.no_grad():
    prediction = pt_model_classification(sample_input)
    predicted_class = torch.argmax(prediction, axis=1).item()

print(f'Predicted Class: {iris.target_names[predicted_class]}')

Predicted Class: virginica


#### Step 17- TensorFlow- Build the MLP Model for Classification

In [103]:
y_train_one_hot = to_categorical(y_train, num_classes=3)
y_test_one_hot = to_categorical(y_test, num_classes=3)

# Create a new model (to avoid sharing weights)
tf_model_classification = keras.Sequential([
    Input(shape=(X_train.shape[1],)),  # Define input layer
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])

# Compile with categorical crossentropy
tf_model_classification.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

tf_model_classification.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

#### Step 18 - TensorFlow - Train the Classification Model

In [106]:
# Train the model
history_classification = tf_model_classification.fit(
    X_train, y_train_one_hot,
    validation_data=(X_test, y_test_one_hot),
    epochs=100,
    batch_size=16,
    verbose=1
)

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.1333 - loss: 1.1268 - val_accuracy: 0.2667 - val_loss: 1.0845
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2417 - loss: 1.0829 - val_accuracy: 0.3000 - val_loss: 1.0439
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3833 - loss: 1.0432 - val_accuracy: 0.5667 - val_loss: 1.0069
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5250 - loss: 1.0047 - val_accuracy: 0.6333 - val_loss: 0.9690
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6250 - loss: 0.9648 - val_accuracy: 0.7333 - val_loss: 0.9300
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7000 - loss: 0.9264 - val_accuracy: 0.7000 - val_loss: 0.8900
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7167 - loss: 0.8844 - val_accuracy: 0.7667 - val_loss: 0.8536
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7500 - loss: 0.8514 - val_accuracy: 0.8000 - val_loss:

#### Step 19 - TensorFlow- Evaluate the Classification Model

In [109]:
# Evaluate the model on test data
# Use one-hot encoded labels for evaluation
loss, accuracy = tf_model_classification.evaluate(X_test, y_test_one_hot)
print(f"Test Accuracy: {accuracy:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.0754
Test Accuracy: 1.0000


# Part 2 - Implementation of Classification- PyTorch Implementation

# Data Preparation
#### The Iris dataset contains measurements of different iris flowers with the goal of classifying them into three species.

#### The data preparation is similar to the regression task:

#### Load the dataset
#### Extract features and target labels
#### Normalize features
#### Split into training and testing sets

In [112]:
# Load Iris dataset
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

In [114]:
# Add target (species)
df_iris['target'] = iris.target

# Features and target
X = df_iris[iris.feature_names].values
y = df_iris['target'].values

In [116]:
# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)


In [118]:
# Train-test split (80/20, stratify to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("First 5 rows of features:\n", X_train[:5])
print("First 5 labels:", y_train[:5])

Training shape: (120, 4)
Testing shape: (30, 4)
First 5 rows of features:
 [[-1.74885626 -0.36217625 -1.34022653 -1.3154443 ]
 [-1.14301691 -1.28296331  0.42173371  0.65903847]
 [ 1.15917263 -0.59237301  0.59224599  0.26414192]
 [-1.14301691  0.09821729 -1.2833891  -1.44707648]
 [-0.41600969 -1.28296331  0.13754657  0.13250973]]
First 5 labels: [0 2 1 0 1]


#### Step 12- Converting the data
#### Data is converted to PyTorch tensors, with targets as long integers for classification.

In [122]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)  # Long tensor for classification
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

#### Step 12: Building the Model
#### The PyTorch classification model is simpler than the regression model:

In [125]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

#### It has one hidden layer with 16 neurons and uses softmax activation for the final layer to output class probabilities.

In [128]:
tf_model_classification = keras.Sequential([
    Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])

In [130]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

# Initialize model
input_size = X_train_t.shape[1]
hidden_size = 16
output_size = len(iris.target_names)

pt_model_classification = MLPClassifier(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model_classification.parameters(), lr=0.01)

In [132]:
num_epochs = 100
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = pt_model_classification(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [0/100], Loss: 1.1137
Epoch [10/100], Loss: 0.8786
Epoch [20/100], Loss: 0.7720
Epoch [30/100], Loss: 0.7200
Epoch [40/100], Loss: 0.6881
Epoch [50/100], Loss: 0.6623
Epoch [60/100], Loss: 0.6393
Epoch [70/100], Loss: 0.6186
Epoch [80/100], Loss: 0.6040
Epoch [90/100], Loss: 0.5949


In [134]:
with torch.no_grad():
    predictions = pt_model_classification(X_test_t)
    predicted_labels = torch.argmax(predictions, axis=1)
    accuracy = (predicted_labels == y_test_t).sum().item() / y_test_t.size(0)

print(f'Classification Accuracy: {accuracy:.4f}')

Classification Accuracy: 0.9667


In [136]:
sample_input = torch.tensor([[6.7, 3.1, 4.9, 1.5]])  # Example from dataset
with torch.no_grad():
    prediction = pt_model_classification(sample_input)
    predicted_class = torch.argmax(prediction, axis=1).item()

print(f'Predicted Class: {iris.target_names[predicted_class]}')

Predicted Class: virginica
